# Lab 12: Generative Adversarial Networks

            **Duration:** 3 hours  
            **Lecture alignment:** Week 12 — GAN objectives and training dynamics  
            **CLO mapping:** CLO-1, CLO-2, CLO-3, CLO-4  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Implement alternating discriminator and generator updates.
- Diagnose mode coverage and unstable adversarial losses.
- Generate low-resolution image samples with a CPU-friendly GAN.

            ## Three-hour activity plan

            - 0–30 min: minimax objective and alternating updates
- 30–90 min: train 2-D multimodal GAN
- 90–125 min: mode-collapse diagnostics
- 125–160 min: image-GAN smoke test
- 160–180 min: stabilization analysis and checks


## Book grounding

            - Goodfellow, Bengio, and Courville, *Deep Learning*, MIT Press, 2016.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20272
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_12")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_12"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 12, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Do you expect discriminator and generator losses to move monotonically? Predict a mode-coverage result and explain why loss alone may mislead.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Adversarial learning on an eight-mode distribution


In [ ]:
centers=torch.stack([torch.tensor([math.cos(k*math.pi/4),math.sin(k*math.pi/4)])*2 for k in range(8)])
def real_points(n):
    ids=torch.randint(0,8,(n,));return centers[ids]+.08*torch.randn(n,2)
class Generator(nn.Module):
    def __init__(self):super().__init__();self.net=nn.Sequential(nn.Linear(4,32),nn.ReLU(),nn.Linear(32,32),nn.ReLU(),nn.Linear(32,2))
    def forward(self,z):return self.net(z)
class Discriminator(nn.Module):
    def __init__(self):super().__init__();self.net=nn.Sequential(nn.Linear(2,32),nn.LeakyReLU(.2),nn.Linear(32,32),nn.LeakyReLU(.2),nn.Linear(32,1))
    def forward(self,x):return self.net(x)
G,D=Generator().to(DEVICE),Discriminator().to(DEVICE);gopt=torch.optim.Adam(G.parameters(),lr=.002,betas=(.5,.9));dopt=torch.optim.Adam(D.parameters(),lr=.002,betas=(.5,.9));g_hist=[];d_hist=[]
for step in range(180 if FAST_MODE else 1200):
    real=real_points(128).to(DEVICE);z=torch.randn(128,4,device=DEVICE);fake=G(z)
    dopt.zero_grad();d_loss=F.binary_cross_entropy_with_logits(D(real),torch.ones(128,1,device=DEVICE)*.9)+F.binary_cross_entropy_with_logits(D(fake.detach()),torch.zeros(128,1,device=DEVICE));d_loss.backward();dopt.step()
    gopt.zero_grad();fake=G(torch.randn(128,4,device=DEVICE));g_loss=F.binary_cross_entropy_with_logits(D(fake),torch.ones(128,1,device=DEVICE));g_loss.backward();gopt.step()
    g_hist.append(g_loss.item());d_hist.append(d_loss.item())
with torch.no_grad():generated=G(torch.randn(1500,4,device=DEVICE)).cpu()
nearest=torch.cdist(generated,centers).argmin(1);counts=torch.bincount(nearest,minlength=8);mode_coverage=int((counts>15).sum())
print({"mode_coverage":mode_coverage,"counts":counts.tolist(),"g_loss":g_hist[-1],"d_loss":d_hist[-1]})


## Activity 2 — A tiny image GAN smoke test


In [ ]:
def real_icons(n):
    labels=torch.randint(0,2,(n,));x=torch.zeros(n,64)
    for i,label in enumerate(labels.tolist()):
        image=torch.zeros(8,8)
        if label==0:image[:,3:5]=1
        else:image[3:5,:]=1
        x[i]=(image+.05*torch.randn_like(image)).clamp(0,1).flatten()
    return x
image_G=nn.Sequential(nn.Linear(12,48),nn.ReLU(),nn.Linear(48,64),nn.Sigmoid()).to(DEVICE)
image_D=nn.Sequential(nn.Linear(64,48),nn.LeakyReLU(.2),nn.Linear(48,1)).to(DEVICE)
igopt=torch.optim.Adam(image_G.parameters(),lr=.003);idopt=torch.optim.Adam(image_D.parameters(),lr=.003)
for _ in range(50 if FAST_MODE else 400):
    real=real_icons(64).to(DEVICE);fake=image_G(torch.randn(64,12,device=DEVICE))
    idopt.zero_grad();dl=F.binary_cross_entropy_with_logits(image_D(real),torch.ones(64,1,device=DEVICE))+F.binary_cross_entropy_with_logits(image_D(fake.detach()),torch.zeros(64,1,device=DEVICE));dl.backward();idopt.step()
    igopt.zero_grad();fake=image_G(torch.randn(64,12,device=DEVICE));gl=F.binary_cross_entropy_with_logits(image_D(fake),torch.ones(64,1,device=DEVICE));gl.backward();igopt.step()
with torch.no_grad():icon_samples=image_G(torch.randn(12,12,device=DEVICE)).cpu().reshape(-1,8,8)


## Activity 3 — Mode coverage and generated-sample diagnostics


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.6));real=real_points(800)
axes[0].scatter(real[:,0],real[:,1],s=5,alpha=.4);axes[0].set_title("Real modes")
axes[1].scatter(generated[:,0],generated[:,1],s=5,alpha=.4);axes[1].scatter(centers[:,0],centers[:,1],marker="x",c="black");axes[1].set_title(f"Generated: {mode_coverage}/8 modes")
axes[2].plot(g_hist,label="G");axes[2].plot(d_hist,label="D");axes[2].legend();axes[2].set_title("Adversarial losses");fig.tight_layout();fig.savefig(ARTIFACT_DIR/"gan_diagnostics.png",dpi=150);plt.show()
fig,axes=plt.subplots(2,6,figsize=(7,3));
for image,ax in zip(icon_samples,axes.flat):ax.imshow(image,cmap="gray",vmin=0,vmax=1);ax.axis("off")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"gan_icons.png",dpi=150);plt.show();torch.save(G.state_dict(),ARTIFACT_DIR/"generator.pt")


## Automated checks


In [ ]:
assert generated.shape==(1500,2) and icon_samples.shape==(12,8,8)
assert torch.isfinite(generated).all() and all(math.isfinite(v) for v in (g_hist[-1],d_hist[-1]))
assert 1<=mode_coverage<=8
assert (ARTIFACT_DIR/"gan_diagnostics.png").exists()
print("All Lab 12 checks passed.")


## Deliverables

                - Correctly detached discriminator update
- Mode-coverage/loss artifact
- Generated image grid and saved generator
- Brief diagnosis of instability or collapse

                Submit the executed notebook and the files created in `/content/artifacts/lab_12/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: condition the image generator and discriminator on icon class labels.")
else:
    print("Extension disabled: conditional GAN, convolutional GAN, or alternative adversarial loss.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
